In [1]:
import pandas as pd
import requests
import time
import json
import os
import sys


from typing import Optional
from tqdm import tqdm
from ast import literal_eval
from urllib.parse import quote

sys.path.append(os.path.abspath("../src"))

from preprocessing import (
    filter_synonym_list, 
    preprocess_chemical,
)
from harmonization import (
    harmonize_automated_classification,
    harmonize_manual_classification,
)

from visualization import (
    compute_metrics_single_method
)

from api import (
    get_cpdat_puc_superclasses_inchikey,
    map_cpdat_to_chemsource
)

In [2]:
classified_drug_library_data_path = "../data/drug_library/validation_data_classified_all_3_methods.csv"
raw_drug_library_smiles_path = "../data/drug_library/20250107_druglib_synonyms_with_smiles.csv"
harmonized_automated = harmonize_automated_classification(classified_drug_library_data_path)
harmonized_manual = harmonize_manual_classification(classified_drug_library_data_path)

drug_library_text = pd.read_csv(classified_drug_library_data_path)
raw_drug_library_smiles = pd.read_csv(raw_drug_library_smiles_path, index_col=0)
raw_drug_library_smiles["synonyms"] = raw_drug_library_smiles["synonyms"].apply(lambda x: literal_eval(x) if pd.notna(x) else [])
raw_drug_library_smiles = raw_drug_library_smiles.drop_duplicates(subset=["compound_name"])
raw_drug_library_smiles["synonyms"] = raw_drug_library_smiles["synonyms"].apply(lambda x: filter_synonym_list(x))
raw_drug_library_smiles["synonyms"] = raw_drug_library_smiles["synonyms"].apply(lambda x: preprocess_chemical(x))
raw_drug_library_smiles_subset = raw_drug_library_smiles[["synonyms", "smiles", "inchikey"]]

drug_library_text["synonyms"] = drug_library_text["synonyms"].apply(lambda x: literal_eval(x) if pd.notna(x) else ())

In [3]:
def merge_on_list_overlap(df1, df2, col1, col2, how="left", priority_col=None):
    """
    Merge two dataframes based on having at least one shared item between list columns.
    
    Priority for selecting best match:
    1. If priority_col is specified: prefer rows where priority_col is not null/NaN
    2. Highest overlap count
    3. First match (ties)
    
    Args:
        df1: First dataframe
        df2: Second dataframe
        col1: Column name in df1 containing lists
        col2: Column name in df2 containing lists
        how: Type of merge ('inner', 'left')
        priority_col: Column name in df2 to prioritize non-null values (e.g., 'inchikey')
        
    Returns:
        Merged dataframe with rows where lists share at least one item
    """
    # Create a copy to avoid modifying originals
    df1 = df1.copy().reset_index(drop=True)
    df2 = df2.copy().reset_index(drop=True)
    
    # Build inverted index: item -> list of (df2_index, original_order)
    # This avoids O(n*m) comparisons
    inverted_index = {}
    for j, row2 in enumerate(df2[col2]):
        if row2:
            items = set(row2) if not isinstance(row2, set) else row2
            for item in items:
                if item not in inverted_index:
                    inverted_index[item] = []
                inverted_index[item].append(j)
    
    # Convert df2 lists to sets once
    df2_sets = [set(row) if row else set() for row in df2[col2]]
    
    # Pre-compute priority column non-null status if specified
    if priority_col and priority_col in df2.columns:
        has_priority = [pd.notna(val) for val in df2[priority_col]]
    else:
        has_priority = None
    
    # Find best match for each row in df1
    best_matches = {}  # {i: (j, overlap_count)}
    
    for i, row1 in enumerate(df1[col1]):
        if not row1:
            continue
        
        list1 = set(row1) if not isinstance(row1, set) else row1
        
        # Find candidate df2 rows using inverted index
        candidate_js = set()
        for item in list1:
            if item in inverted_index:
                candidate_js.update(inverted_index[item])
        
        if not candidate_js:
            continue
        
        # Find best match among candidates
        best_j = None
        best_overlap = 0
        best_has_priority = False
        
        for j in sorted(candidate_js):  # sorted ensures first wins ties
            overlap_count = len(list1 & df2_sets[j])
            
            if overlap_count > 0:
                current_has_priority = has_priority[j] if has_priority else False
                
                # Determine if this is a better match
                # Priority order: has_priority > overlap_count > first
                is_better = False
                
                if best_j is None:
                    is_better = True
                elif has_priority:
                    # If current has priority and best doesn't, current wins
                    if current_has_priority and not best_has_priority:
                        is_better = True
                    # If both have same priority status, use overlap count
                    elif current_has_priority == best_has_priority:
                        if overlap_count > best_overlap:
                            is_better = True
                else:
                    # No priority column, just use overlap count
                    if overlap_count > best_overlap:
                        is_better = True
                
                if is_better:
                    best_overlap = overlap_count
                    best_j = j
                    best_has_priority = current_has_priority
        
        if best_j is not None:
            best_matches[i] = (best_j, best_overlap)
    
    # Build result dataframe
    if how == "inner":
        if not best_matches:
            return pd.DataFrame()
        
        matched_i = list(best_matches.keys())
        matched_j = [best_matches[i][0] for i in matched_i]
        
        result = df1.loc[matched_i].reset_index(drop=True)
        df2_matched = df2.loc[matched_j].reset_index(drop=True)
        
        # Add all df2 columns including the list column (renamed with _matched suffix)
        for col in df2.columns:
            result[col + "_matched"] = df2_matched[col].values
        
        result["_overlap_count"] = [best_matches[i][1] for i in matched_i]
        return result
    
    elif how == "left":
        result = df1.copy()
        
        # Pre-allocate columns with object dtype for lists
        for col in df2.columns:
            result[col + "_matched"] = pd.Series([None] * len(result), dtype=object)
        result["_overlap_count"] = 0
        
        # Fill in matches row by row (needed for list values)
        if best_matches:
            for i, (j, overlap_count) in best_matches.items():
                for col in df2.columns:
                    result.at[i, col + "_matched"] = df2.at[j, col]
                result.at[i, "_overlap_count"] = overlap_count
        
        return result
    
    else:
        raise ValueError(f"Merge type '{how}' not implemented. Use 'inner' or 'left'.")


# Example usage:
# Merge drug_library_text and raw_drug_library_smiles_subset based on shared synonyms
# Priority: prefer matches with non-null inchikey
merged_df = merge_on_list_overlap(
    drug_library_text, 
    raw_drug_library_smiles_subset, 
    col1="synonyms",  # list column in drug_library_text
    col2="synonyms",  # list column in raw_drug_library_smiles_subset
    how="left",
    priority_col="inchikey"  # prioritize matches with non-null inchikey
)


# Fill in missing inchikeys from matched data manually
merged_df.at[2389, "synonyms_matched"] = ['(+)-Tetrabenazine', '1026016-83-0', '(+)-TBZ', '(3R,11bR)-TBZ', '(3R,11bR)-Tetrabenazine', 'Tetrabenazine (+)-', '(3R,11bR)-3-isobutyl-9,10-dimethoxy-3,4,6,7-tetrahydro-1H-pyrido[2,1-a]isoquinolin-2(11bH)-one', '69ENL3U6BF', 'CHEMBL61636', 'CHEBI:64029', '(3R,11bR)-9,10-dimethoxy-3-(2-methylpropyl)-1,3,4,6,7,11b-hexahydrobenzo[a]quinolizin-2-one', '(+)-Ro 1-9569', 'Tetrabenazine, (+)-', '(3R,11bR)-3-Isobutyl-9,10-dimethoxy-1,3,4,6,7,11b-hexahydro-pyrido[2,1-a]isoquinolin-2-one', '(3R,11bR)-9,10-dimethoxy-3-(2-methylpropyl)-1,3,4,6,7,11b-hexahydro-2H-pyrido[2,1-a]isoquinolin-2-one', '(3R,11bR)-9,10-dimethoxy-3-isobutyl-1,3,4,6,7,11b-hexahydro-2H-pyrido[2,1-a]isoquinolin-2-one', '(R,R)-1,3,4,6,7,11b-hexahydro-9,10-dimethoxy-3-(2-methylpropyl)-2H-benzo[a]quinolizin-2-one', 'UNII-69ENL3U6BF', 'Tetrabenazine R,R-form [MI]', 'Tetrabenazine ((+)-)', 'ZINC751', 'SCHEMBL340173', 'HY-B0590B', 'BDBM50048891', 'AKOS025290764', 'CS-5827', 'Tetrabenazine, >=98% (HPLC), solid', '(3R,11bR)-1,3,4,6,7,11b-Hexahydro-9,10-dimethoxy-3-(2-methylpropyl)-2H-benzo(a)quinolizin-2-one', '2H-Benzo(a)quinolizin-2-one, 1,3,4,6,7,11b-hexahydro-9,10-dimethoxy-3-(2-methylpropyl)-, (3R,11bR)-', 'AC-22623', 'AS-74541', 'D97714', 'P10603', '058T468', 'A896752', 'BRD-K24125544-001-01-3', 'Q27132977', '(3R,11bR)-1,3,4,6,7,11b-Hexahydro-9,10-dimethoxy-3-(2-methylpropyl)-2H-benzo[a]quinolizin-2-one;Tetrabenazine (+)-', '(3R,11bR)-3-isobutyl-9,10-dimethoxy-1,3,4,6,7,11b-hexahydro-2H-pyrido[2,1-a]isoquinolin-2-one', '(3R,11bR)-9,10-dimethoxy-3-(2-methylpropyl)-1H,2H,3H,4H,6H,7H,11bH-pyrido[2,1-a]isoquinolin-2-one']
merged_df.at[2389, "inchikey_matched"] = "MKJIEFSOBYUXJB-HOCLYGCPSA-N"
merged_df.at[2389, "smiles_matched"] = "COc1cc2c(cc1OC)C1CC(=O)C(CC(C)C)CN1CC2"

merged_df.at[4842, "synonyms_matched"] = ['2-Mercaptobenzothiazole', '149-30-4', '2-Benzothiazolethiol', 'Captax', 'Benzo[d]thiazole-2(3H)-thione', 'Benzo[d]thiazole-2-thiol', 'Benzothiazolethiol', 'MERCAPTOBENZOTHIAZOLE', 'Benzothiazole-2-thiol', '1,3-Benzothiazole-2-thiol', '2(3H)-Benzothiazolethione', 'Dermacid', 'Mertax', 'Rotax', 'Accelerator M', '2-MBT', 'Sulfadene', 'Kaptax', 'Thiotax', 'Rokon', '118090-09-8', 'Vulkacit M', 'Ekagom G', 'Accel M', 'Mebetizole', 'Mebithizol', 'Kaptaks', 'Nuodeb 84', 'Soxinol M', 'Vulkacit Mercapto', 'Pneumax MBT', '2-Mercaptobenzthiazole', 'Royal MBT', 'Mercaptobenzothiazol', 'Mercaptobenzthiazole', 'Vulkacit Mercapto/C', '2-Mercptobenzothiazole', 'Pennac mbt powder', 'Benzothiazole-2-thione', '2-Benzothiazolinethione', 'Nuodex 84', 'Usaf gy-3', 'MBT', 'Nocceler M', 'Usaf xr-29', 'Benzothiazole, mercapto-', '1,3-Benzothiazol-2-yl hydrosulfide', '2-Benzothiazolyl mercaptan', '2-Merkaptobenzotiazol', '2-Merkaptobenzthiazol', 'AG 63', 'benzothiazolyl mercaptan', 'Perkacit MBT', '3H-1,3-benzothiazole-2-thione', '2-Benzothiazolethiol(9CI)', 'Mercaptobenzothiazole (VAN)', '2-sulfanyl-1,3-benzothiazole', '2-Mercapto benzothiazole', '2-mercapto-benzothiazole', 'CHEBI:34292', 'NCI-C56519', '2-benzothiazolthiol', 'Accelerator mercapto', '2-Mercaptobenzothioazole', 'MFCD00005781', '5RLR54Z22K', 'mebetizol', '2-Mercaptobenzothiazole (in liquid mixtures)', 'DTXSID1020807', 'NSC2041', 'NCGC00091643-07', 'NCGC00091643-08', 'DSSTox_CID_807', 'Kaptax [Czech]', 'DSSTox_RID_75799', 'DSSTox_GSID_20807', '2-thiobenzothiazole', '2(3H)-Benzothiazolethione, potassium salt', 'Caswell No. 541', 'pennac mbt', 'Thiot ax', 'captax, zinc salt', 'CAS-149-30-4', 'CCRIS 891', 'captax, sodium salt', '2-Merkaptobenzthiazol [Czech]', '2-Merkaptobenzotiazol [Polish]', 'captax, potassium salt', 'HSDB 4025', '2-Sulfanylbenzothiazole', 'NSC 2041', 'captax, lead(+2) salt', 'EINECS 205-736-8', 'captax, cobalt(+2) salt', 'captax, copper(+2) salt', 'captax, silver(+1) salt', 'captax, bismuth(+3) salt', 'EPA Pesticide Chemical Code 051701', 'UNII-5RLR54Z22K', 'captax, mercury (+2) salt', 'Drmacid', 'AI3-00985', 'thiotax(tm)', 'rokon(r)', 'MBT, captax', 'mercapto-benzothiazole', '2-Benzothiazolethione', '2-mercaptobenzothiazol', '2-mercapto-benzthiazole', 'Spectrum_001669', 'SpecPlus_000728', '2-thiocarbamidothiophenol', '155-04-4', '57948-09-1', 'Spectrum2_001666', 'Spectrum3_001665', 'Spectrum4_000628', 'Spectrum5_001400', 'mercapto(2-)benzothiazole', '2(3H)-Benzothiazoletione', 'benzo[d]thiazole-2-thione', 'Epitope ID:116044', 'EC 205-736-8', 'Benzothiazole, 2-mercapto-', 'SCHEMBL23237', '1,3-Benzothiazole-2-thione', 'BSPBio_003449', 'KBioGR_001216', 'KBioSS_002149', 'BIDD:ER0373', 'DivK1c_006824', 'SPECTRUM1504225', '2-MercaptobenzothiazoleDermacid', 'SPBio_001851', '2-Mercaptobenzothiazole, 97%', '2-Sulphanyl-1,3-benzothiazole', 'CHEMBL111654', '155-04-4 (zinc salt)', 'WLN: T56 BN DSJ CSH', 'Vulkacit M, vulkacit merkapto/c', 'KBio1_001768', 'KBio2_002149', 'KBio2_004717', 'KBio2_007285', 'KBio3_002669', '2-Mercaptobenzothiazole (2-MBT)', '1,3-benzothiazol-2-ylhydrosulfide', '7778-70-3 (potassium salt)', 'AMY23224', 'NSC-2041', 'Tox21_113450', 'Tox21_400016', '1,3-Benzothiazol-2-yl hydrosulphide', 'BDBM50444459', 'c1019', 'CCG-39092', 'STK499589', 'ZINC18098783', 'AKOS000119128', 'AKOS002337495', '1,3-Benzothiazol-2-yl hydrosulfide #', 'CS-W017829', 'DB11496', 'FS-1801', 'HY-W017113', '4162-43-0 (copper(+2) salt)', 'NCGC00091643-01', 'NCGC00091643-02', 'NCGC00091643-04', 'NCGC00091643-05', 'NCGC00091643-06', 'NCGC00091643-09', 'NCGC00091643-10', 'NCGC00091643-12', 'AC-11606', 'DB-042988', 'FT-0612758', 'FT-0699702', 'M0055', 'M0247', '49M304', 'D70518', '2-Mercaptobenzothiazole, technical, >=90% (T)', 'AB00053232-04', 'A808877', 'A927195', 'AE-641/31369054', 'Q904160', 'Q-200294', 'BRD-K55160477-001-02-1', 'BRD-K55160477-001-03-9', 'F3066-0005', 'Z1250100728', '27157-85-3']
merged_df.at[4842, "inchikey_matched"] = "YXIWHUQXZSMYRE-UHFFFAOYSA-N"
merged_df.at[4842, "smiles_matched"] = "S=c1[nH]c2ccccc2s1"

merged_df.at[4189, "synonyms_matched"] = ['Prednisolone Tebutate', '7681-14-3', 'Hydeltra-T.B.A.', 'Prednisolone Butylacetate', 'Prednisolone 21-tert-butylacetate', 'EINECS 231-661-5', 'UNII-1V7A1U282K', 'Anhydrous prednisolone tebutate', 'CHEBI:8381', '[2-[(8S,9S,10R,11S,13S,14S,17R)-11,17-dihydroxy-10,13-dimethyl-3-oxo-7,8,9,11,12,14,15,16-octahydro-6H-cyclopenta[a]phenanthren-17-yl]-2-oxoethyl] 3,3-dimethylbutanoate', '1V7A1U282K', 'Codelcortone TBA', 'Predalone T.B.A.', 'Prednisolone tebutate [USP]', 'Hydeltra-TBA (TN)', 'Prednisolone butyl acetete', 'DSSTox_CID_3505', '11beta,17,21-Trihydroxypregna-1,4-diene-3,20-dione 21-(3,3-dimethylbutyrate)', 'DSSTox_RID_97549', 'DSSTox_GSID_23505', 'SCHEMBL40845', 'CHEMBL1200909', 'DTXSID8023505', 'Prednisolone tebutate (JAN/USP)', 'Prednisolone tertiary butylacetate', 'ZINC4097474', 'Tox21_113662', 'HY-U00098', 'CS-7134', 'DB14632', 'Pregna-1,4-diene-3,20-dione, 21-(3,3-dimethyl-1-oxobutoxy)-11,17-dihydroxy-, (11beta)-', 'NCGC00249886-01', 'Pregna-1,4-diene-3,20-dione, 11,17-dihydroxy-21-((3,3-dimethyl-1-oxobutyl)oxy)-, (11beta)-', 'CAS-7681-14-3', 'C08182', 'D00982', '681P143', 'Q27108064', '(11beta)-11,17-dihydroxy-3,20-dioxopregna-1,4-dien-21-yl 3,3-dimethylbutanoate', '[2-[(8S,9S,10R,11S,13S,14S,17R)-11,17-dihydroxy-10,13-dimethyl-3-oxo-7,8,9,11,12,14,15,16-octahydro-6H-cyclopenta[a]phenanthren-17-yl]-2-oxo-ethyl] 3,3-dimethylbutanoate']
merged_df.at[4189, "inchikey_matched"] = "HUMXXHTVHHLNRO-UHFFFAOYSA-N"
merged_df.at[4189, "smiles_matched"] = "CC(C)(C)CC(=O)OCC(=O)C1(O)CCC2C3CC=C4CC(=O)C=CC4(C)C3C(O)CC21C"

merged_df.at[4440, "synonyms_matched"] = ['Anagrelide (hydrochloride)', '58579-51-4', 'BL4162A']
merged_df.at[4440, "inchikey_matched"] = "OTBXOEAOVRKTNQ-UHFFFAOYSA-N"
merged_df.at[4440, "smiles_matched"] = "Oc1cn2c(n1)Nc1ccc(Cl)c(Cl)c1C2"


merged_df.at[3109, "synonyms_matched"] = ['(S)-Tenofovir', '147127-19-3', '(S)-GS 1278', '(S)-PMPA', '(S)-TDF', '(S)-(((1-(6-Amino-9H-purin-9-yl)propan-2-yl)oxy)methyl)phosphonic acid', '(S)-9-(2-Phosphonylmethoxypropyl)adenine', '[(2S)-1-(6-aminopurin-9-yl)propan-2-yl]oxymethylphosphonic acid', '({[(2S)-1-(6-amino-9H-purin-9-yl)propan-2-yl]oxy}methyl)phosphonic acid', 'Phosphonic acid, (((1S)-2-(6-amino-9H-purin-9-yl)-1-methylethoxy)methyl)-', 'MFCD21604708', 'Phosphonic acid, [[(1S)-2-(6-amino-9H-purin-9-yl)-1-methylethoxy]methyl]-', 'CHEMBL483891', 'SCHEMBL2040329', 'AMY11188', 'ZINC2020246', '(S)-(((1-(6-Amino-9H-purin-9-yl)propan-2-yl)oxy)methyl)phosphonicacid', 'AKOS016844118', 'HY-W074930', 'Phosphonic acid, P-[[(1S)-2-(6-amino-9H-purin-9-yl)-1-methylethoxy]methyl]-', 'AS-73240', '(s)-9-[2-(phosphonomethoxy)propyl]adenine', 'CS-0111892', 'O10188', 'J-008317', '[(1S)-2-(6-aminopurin-9-yl)-1-methyl-ethoxy]methylphosphonic acid']
merged_df.at[3109, "inchikey_matched"] ="SGOIRFVFHAKUTI-LURJTMIESA-N"
merged_df.at[3109, "smiles_matched"] = "CC(Cn1cnc2c(=N)[nH]cnc21)OC[PH](=O)(=O)O"

merged_df.at[4675, "synonyms_matched"] = ['guanine', '73-40-5', '2-Amino-6-hydroxypurine', 'Guanin', '2-Aminohypoxanthine', 'Mearlmaid', 'Pearl essence', 'Guanine enol', 'Stella Polaris', 'Dew Pearl', 'Natural pearl essence', 'Natural white 1', '6H-Purin-6-one, 2-amino-1,7-dihydro-', '2-Amino-6-purinol', 'C.I. Natural White 1', 'CI Natural white 1', 'Hypoxanthine, 2-amino-', '6-Hydroxy-2-aminopurine', '2-Amino-1,7-dihydro-6H-purin-6-one', '2-amino-1,9-dihydro-6H-purin-6-one', '2-amino-1,7-dihydropurin-6-one', 'CI 75170', 'C.I. 75170', 'HSDB 2127', 'Mearlmaid AA', '6H-Purin-6-one, 2-amino-1,9-dihydro-', '2-amino-9H-purin-6-ol', 'AI3-24393', '2-AMINO-3H-PURIN-6(7H)-ONE', '2-Amino-6-hydroxy-1H-purine', 'CHEBI:16235', '2-amino-1,9-dihydropurin-6-one', '6H-purin-6-one, 2-amino-3,7-dihydro-', 'CHEMBL219568', '2-amino-1h-purin-6(9h)-one', '5Z93L87A1R', '2-Amino-1,9-dihydro-purin-6-one', '2-amino-6,7-dihydro-3H-purin-6-one', 'GUA', 'GUN', '2-amino-6-hydroxypurin', 'EINECS 200-799-8', 'MFCD00071533', 'UNII-5Z93L87A1R', '7H-Purin-6-ol, 2-amino-', '9H-Purin-6-ol, 2-amino-', '9h-guanine', 'Guanine, BioUltra', 'Guanine,(S)', 'Guanine (8CI)', '2-amino-6-oxypurin', '2-amino-Hypoxanthine', 'Guanine, 98%', '1H-Purin-6-ol, 2,3-dihydro-2-imino-', 'Aciclovir EP Impurity B', '2-amino-7H-purin-6-ol', 'bmse000090', 'Epitope ID:140098', 'SCHEMBL5259', 'Oprea1_875298', '66224-64-4', 'GTPL4556', 'DTXSID9052476', 'SCHEMBL16389311', 'C.i. no. 75170 (guanine)', 'ZINC895129', 'ALBB-025935', 'BCP26793', 'HY-Y1055', 'Valacyclovir hydrochloride, guanine-', 'BBL009290', 'BBL009641', 'BDBM50200094', 's4888', 'STK297804', 'STK801924', 'AKOS000118904', 'AKOS001426592', 'AKOS005139176', 'AKOS016002094', 'Valganciclovir hydrochloride impurity b', 'AC-4743', 'AM81389', 'CS-6269', 'DB02377', '2-amino-3,7-dihydro-6H-purin-6-one', '2-amino-6,9-dihydro-1H-purin-6-one', 'NCGC00246975-01', '66224-61-1', '66224-63-3', '71660-31-6', '71660-36-1', 'AS-10918', 'Guanine, Vetec(TM) reagent grade, 99%', 'NCI60_012450', 'DB-015937', 'FT-0611249', 'G0169', 'EN300-21473', 'A15593', 'C00242', '2-Amino-1,7-dihydro-6H-purin-6-one (Guanine)', '6H-Purin-6-one, 2-amino-1,7-dihydro- (9CI)', 'A866095', 'Q169313', 'W-104453', 'F8880-3425', 'Z256709612', '3D215030-CD54-4835-A5F4-F00F86B90978', 'Guanine, United States Pharmacopeia (USP) Reference Standard', 'Phosphonium,[3-(dimethylamino)propyl]triphenyl bromide hydrobromide', 'Guanine, Pharmaceutical Secondary Standard; Certified Reference Material', 'Acyclovir EP Impurity B; Valaciclovir EP Impurity A; Valaciclovir USP Related Compound A; 2-Amino-1,9-dihydro-6H-purin-6-one']
merged_df.at[4675, "inchikey_matched"] = "UYTPUPDQBNUYGX-UHFFFAOYSA-N"
merged_df.at[4675, "smiles_matched"] = "N=c1[nH]c(=O)c2[nH]cnc2[nH]1"

# add feature_id col at front
merged_df["FEATURE_ID"] = merged_df.index
merged_df = merged_df[["FEATURE_ID"] + [col for col in merged_df.columns if col != "FEATURE_ID"]]

In [6]:
input_inchikeys = merged_df["inchikey_matched"].tolist()
feature_ids = merged_df["FEATURE_ID"].tolist()
output_cpdat_data = []
for i, item in tqdm(enumerate(input_inchikeys), total=len(input_inchikeys)):
    cpdat_superclasses = get_cpdat_puc_superclasses_inchikey(item)
    output_cpdat_data.append({
        "FEATURE_ID": feature_ids[i],
        "INCHIKEY": item,
        "CPDAT_SUPERCLASSES": cpdat_superclasses
    })


  2%|▏         | 77/4953 [01:15<1:14:12,  1.10it/s]

InChIKey 'ALHBJBCQLJZYON-ZQDZILKHSA-N' not found in PubChem


  6%|▌         | 294/4953 [05:09<59:09,  1.31it/s]  

InChIKey 'GMPZPHGHNDMRKL-RZDIXWSQSA-N' not found in PubChem


  6%|▋         | 314/4953 [05:30<1:03:15,  1.22it/s]

...

 95%|█████████▍| 4681/4953 [1:18:07<02:32,  1.79it/s]

Error fetching CID (attempt 1/3): HTTPSConnectionPool(host='pubchem.ncbi.nlm.nih.gov', port=443): Max retries exceeded with url: /rest/pug/compound/inchikey/VCKUSRYTPJJLNI-CQSZACIVSA-N/cids/JSON (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x176172990>: Failed to establish a new connection: [Errno 61] Connection refused'))


 95%|█████████▍| 4682/4953 [1:18:09<03:17,  1.37it/s]

Error fetching CID (attempt 1/3): HTTPSConnectionPool(host='pubchem.ncbi.nlm.nih.gov', port=443): Max retries exceeded with url: /rest/pug/compound/inchikey/VEBVPUXQAPLADL-POYOOMFHSA-N/cids/JSON (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x175600190>: Failed to establish a new connection: [Errno 61] Connection refused'))
Error fetching CPDat data (attempt 1/3): HTTPSConnectionPool(host='pubchem.ncbi.nlm.nih.gov', port=443): Max retries exceeded with url: /sdq/sdqagent.cgi?infmt=json&outfmt=json&query=%7B%22select%22%3A%20%22%2A%22%2C%20%22collection%22%3A%20%22cpdat%22%2C%20%22where%22%3A%20%7B%22ands%22%3A%20%5B%7B%22cid%22%3A%20%22442042%22%7D%5D%7D%7D (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x175e59f90>: Failed to establish a new connection: [Errno 61] Connection refused'))
Error fetching CPDat data (attempt 1/3): HTTPSConnectionPool(host='pubchem.ncbi.nlm.nih.gov', port=443): Max retries exceeded with url: /sdq/

100%|██████████| 4953/4953 [1:21:27<00:00,  1.01it/s]


In [ ]:
merged_data_with_cpdat_from_inchis = pd.merge(
    merged_df, 
    pd.DataFrame(output_cpdat_data),
    on="FEATURE_ID",
    how="left"
)

In [9]:
# merged_data_with_cpdat_from_inchis.to_csv("../data/drug_library/drug_library_data_with_cpdat_from_inchikeys.csv", index=False)
merged_data_with_cpdat_from_inchis = pd.read_csv("../data/drug_library/drug_library_data_with_cpdat_from_inchikeys.csv")
merged_data_with_cpdat_from_inchis["CPDAT_SUPERCLASSES"] = merged_data_with_cpdat_from_inchis["CPDAT_SUPERCLASSES"].apply(lambda x: literal_eval(x) if pd.notna(x) else None)

all_cpdat_superclasses = []
for superclasses in merged_data_with_cpdat_from_inchis["CPDAT_SUPERCLASSES"].dropna():
    for superclass in superclasses:
        all_cpdat_superclasses.append(superclass)

cpdat_superclass_counts = {}
for superclass in all_cpdat_superclasses:
    if superclass not in cpdat_superclass_counts:
        cpdat_superclass_counts[superclass] = 0
    cpdat_superclass_counts[superclass] += 1

print(cpdat_superclass_counts)

# Filter out 'Medical/dental' superclass (observed only 2 times, not enough
# for additional category analysis)

merged_data_with_cpdat_from_inchis["CPDAT_SUPERCLASSES"] = merged_data_with_cpdat_from_inchis["CPDAT_SUPERCLASSES"].apply(lambda x: [sc for sc in x if sc != 'Medical/dental'] if x is not None else x)

{'Vehicle': 20, 'Personal care': 164, 'Construction and building materials': 5, 'Home maintenance': 16, 'Cleaning products and household care': 61, 'Specialty occupational products': 1, 'Landscape/yard': 7, 'Laboratory supplies': 9, 'Pet care': 31, 'Pesticides': 27, 'Raw materials': 20, 'Arts and crafts/office supplies': 12, 'Manufactured formulations': 7, 'Furniture and furnishings': 5, 'Electronics/small appliances': 6, 'Cleaning and safety': 4, 'Medical/dental': 2, 'Batteries': 1, 'Cons. electronics, mech. appliances, and machinery': 1}


In [6]:
cpdat_from_inchi_data_filtered = merged_data_with_cpdat_from_inchis[(merged_data_with_cpdat_from_inchis["CPDAT_SUPERCLASSES"].notna()) & (merged_data_with_cpdat_from_inchis["CPDAT_SUPERCLASSES"].str.len() > 0)]
cpdat_from_name_data = pd.read_csv("../data/drug_library/validation_data_cpdat_puc_superclasses.csv")
cpdat_from_name_data["cpdat_puc_superclasses"] = cpdat_from_name_data["cpdat_puc_superclasses"].apply(lambda x: literal_eval(x) if pd.notna(x) else None)
cpdat_from_name_data["cpdat_puc_superclasses"] = cpdat_from_name_data["cpdat_puc_superclasses"].apply(lambda x: [sc for sc in x if sc != 'Medical/dental'] if x is not None else x)
cpdat_from_name_data_filtered = cpdat_from_name_data[(cpdat_from_name_data["cpdat_puc_superclasses"].notna()) & (cpdat_from_name_data["cpdat_puc_superclasses"].str.len() > 0)]

In [10]:
feature_ids_with_inchikeys = set(merged_data_with_cpdat_from_inchis[merged_data_with_cpdat_from_inchis["INCHIKEY"].notna()]["FEATURE_ID"].tolist())
cpdat_from_name_data_filtered_with_inchikeys = cpdat_from_name_data_filtered[cpdat_from_name_data_filtered["feature_id"].isin(feature_ids_with_inchikeys) == True]

In [11]:
cpdat_from_inchis_final = merged_data_with_cpdat_from_inchis[["FEATURE_ID","name_used", "synonyms","INCHIKEY","CPDAT_SUPERCLASSES"]] 
cpdat_from_inchis_final["CPDAT_HARMONIZED"] = cpdat_from_inchis_final["CPDAT_SUPERCLASSES"].apply(lambda x: map_cpdat_to_chemsource(x))
cpdat_from_inchis_final = pd.merge(cpdat_from_inchis_final, harmonized_automated[["FEATURE_ID","DEEPSEEK_RAG","GPT_NO_RAG","GPT_RAG","SEARCH_GPT"]], on="FEATURE_ID", how="left")
cpdat_from_inchis_final = cpdat_from_inchis_final.rename(columns={"name_used":"NAME_USED","synonyms":"SYNONYMS",})


cpdat_from_name_data_updated = cpdat_from_name_data
cpdat_from_name_data_updated = cpdat_from_name_data_updated.rename(columns={"feature_id":"FEATURE_ID", "name_used":"NAME_USED", "synonyms":"SYNONYMS", "cpdat_puc_superclasses":"CPDAT_SUPERCLASSES"})
drug_library_text_updated = drug_library_text
drug_library_text_updated["FEATURE_ID"] = drug_library_text_updated.index
drug_library_text_updated = drug_library_text_updated[["FEATURE_ID","name_used","synonyms"]]
drug_library_text_updated = drug_library_text_updated.rename(columns={"name_used":"NAME_USED", "synonyms":"SYNONYMS"})
cpdat_from_name_data_updated = cpdat_from_name_data_updated[["FEATURE_ID", "CPDAT_SUPERCLASSES"]]
cpdat_from_name_data_updated["CPDAT_HARMONIZED"] = cpdat_from_name_data_updated["CPDAT_SUPERCLASSES"].apply(lambda x: map_cpdat_to_chemsource(x))
cpdat_from_name_data_updated = pd.merge(cpdat_from_name_data_updated, harmonized_automated[["FEATURE_ID","DEEPSEEK_RAG","GPT_NO_RAG","GPT_RAG","SEARCH_GPT"]], on="FEATURE_ID", how="left")
cpdat_from_names_final = pd.merge(drug_library_text_updated, cpdat_from_name_data_updated,on="FEATURE_ID", how="left")


cpdat_from_names_final_filtered = cpdat_from_names_final[cpdat_from_names_final["CPDAT_HARMONIZED"].notna()]
cpdat_from_inchis_final_filtered = cpdat_from_inchis_final[cpdat_from_inchis_final["CPDAT_HARMONIZED"].notna()]

# take union across rows
combined_cpdat_data = pd.concat([cpdat_from_inchis_final_filtered, cpdat_from_names_final_filtered], ignore_index=True)
combined_cpdat_data_final = combined_cpdat_data.groupby("FEATURE_ID").agg("first").reset_index()

# combine first cpdat with inchikeys, then cpdat from names without inchikey
cpdat_from_names_final_filtered_no_inchikeys = cpdat_from_names_final_filtered[cpdat_from_names_final_filtered["FEATURE_ID"].isin(feature_ids_with_inchikeys) == False]
special_combined_cpdat_data = pd.concat([cpdat_from_inchis_final_filtered, cpdat_from_names_final_filtered_no_inchikeys], ignore_index=True)
special_combined_cpdat_data_final = special_combined_cpdat_data.groupby("FEATURE_ID").agg("first").reset_index()

/var/folders/7f/td1j1ghj1f77gp6tfcmnv2lw0000gn/T/ipykernel_10061/3162878138.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cpdat_from_inchis_final["CPDAT_HARMONIZED"] = cpdat_from_inchis_final["CPDAT_SUPERCLASSES"].apply(lambda x: map_cpdat_to_chemsource(x))


In [12]:
cpdat_from_names_final.to_csv("../data/drug_library/cpdat_from_names_final.csv", index=False)
cpdat_from_inchis_final.to_csv("../data/drug_library/cpdat_from_inchis_final.csv", index=False)
combined_cpdat_data_final.to_csv("../data/drug_library/combined_1_cpdat_data_final.csv", index=False)
special_combined_cpdat_data_final.to_csv("../data/drug_library/combined_2_cpdat_data_final.csv", index=False)